In [0]:
# ============================================================
# NOTEBOOK 04 — RANDOM FOREST + ÉVALUATION
# Section 4.3 + Section 5 du papier Belcastro et al. (2016)
# ============================================================

# CELLULE 1 — Initialisation
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler, Imputer
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, \
                                   BinaryClassificationEvaluator
import pandas as pd

spark = SparkSession.builder \
    .appName("FlightDelay_Notebook04") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

print(f"✅ Spark {spark.version} prêt !")


In [0]:
# ============================================================
# CELLULE 2 — Features CORRIGÉES
# Garder seulement les colonnes numériques
# Sky et WType sont des strings → exclus
# ============================================================

METEO_VARS_NUM = ["Temp", "WSpd", "Vis"]  # 3 vars clés seulement
SLOTS          = range(3)                  # 3 slots au lieu de 13

FEATURES_ORIG = [f"orig_{v}_{i}h"
                 for i in SLOTS for v in METEO_VARS_NUM]
FEATURES_DEST = [f"dest_{v}_{i}h"
                 for i in SLOTS for v in METEO_VARS_NUM]
ALL_FEATURES  = FEATURES_ORIG + FEATURES_DEST

print(f"📋 Features : {len(ALL_FEATURES)}")

In [0]:
# ============================================================
# CELLULE 3 — Pipeline ML
# Imputer → VectorAssembler → RandomForest
# Section 4.3 : RF avec 100 arbres, maxDepth=10
# ============================================================

def build_pipeline(label_col, features):
    imputer = Imputer(
        inputCols=features,
        outputCols=[f"{c}_imp" for c in features],
        strategy="mean"
    )
    features_imp = [f"{c}_imp" for c in features]

    assembler = VectorAssembler(
        inputCols=features_imp,
        outputCol="features",
        handleInvalid="skip"
    )

    rf = RandomForestClassifier(
        labelCol=label_col,
        featuresCol="features",
        numTrees=50,
        maxDepth=10,
        maxBins=32,
        seed=42
    )
    return Pipeline(stages=[imputer, assembler, rf])

In [0]:
# ============================================================
# CELLULE 4 — Fonction d'évaluation
# Métriques papier Section 2.4 :
# Acc, Rec_o (on-time recall), Rec_d (delayed recall)
# ============================================================

def evaluate(predictions, label_col):
    """
    Calcule Acc, Rec_o, Rec_d
    selon les définitions du papier (Équations 1 et 2)
    """
    eval_acc = MulticlassClassificationEvaluator(
        labelCol=label_col,
        predictionCol="prediction",
        metricName="accuracy"
    )

    # TP, TN, FP, FN depuis la matrice de confusion
    tp = predictions.filter(
        (col(label_col) == 0) & (col("prediction") == 0)
    ).count()
    tn = predictions.filter(
        (col(label_col) == 1) & (col("prediction") == 1)
    ).count()
    fp = predictions.filter(
        (col(label_col) == 1) & (col("prediction") == 0)
    ).count()
    fn = predictions.filter(
        (col(label_col) == 0) & (col("prediction") == 1)
    ).count()

    acc   = (tp + tn) / (tp + tn + fp + fn) \
            if (tp + tn + fp + fn) > 0 else 0
    rec_o = tp / (tp + fn) if (tp + fn) > 0 else 0
    rec_d = tn / (tn + fp) if (tn + fp) > 0 else 0

    return {
        "Acc":   round(acc   * 100, 1),
        "Rec_o": round(rec_o * 100, 1),
        "Rec_d": round(rec_d * 100, 1),
        "TP": tp, "TN": tn, "FP": fp, "FN": fn
    }


In [0]:
# CELLULE 5 — Corrigée avec del model + sans save

OUTPUT = "/Volumes/workspace/default/outputs/"
configs = [
    ("D1_th15", "label_15"),
    ("D2_th15", "label_15"),
    ("D3_th15", "label_15"),
    # ("D4_th15", "label_15"),
    ("D1_th60", "label_60"),
    ("D2_th60", "label_60"),
    ("D3_th60", "label_60"),
    ("D4_th60", "label_60"),
]

all_results = []
model = None  # initialiser

for name, label_col in configs:
    print(f"\n{'='*55}")
    print(f"  ▶ {name}  —  label : {label_col}")
    print(f"{'='*55}")

    df_train = spark.read.parquet(f"{OUTPUT}{name}_train/")
    df_test  = spark.read.parquet(f"{OUTPUT}{name}_test/")

    existing = df_train.columns
    features = [f for f in ALL_FEATURES if f in existing]
    print(f"  Features : {len(features)}")

    if len(features) == 0:
        continue

    pipeline = build_pipeline(label_col, features)

    # ✅ Vider le cache AVANT chaque fit
    if model is not None:
        del model
    
    print(f"  🔧 Entraînement... (train={df_train.count():,})")
    model = pipeline.fit(df_train)

    print(f"  🔍 Prédiction... (test={df_test.count():,})")
    predictions = model.transform(df_test)
    metrics     = evaluate(predictions, label_col)

    print(f"  ✅ Acc   = {metrics['Acc']}%")
    print(f"  ✅ Rec_o = {metrics['Rec_o']}%")
    print(f"  ✅ Rec_d = {metrics['Rec_d']}%")

    all_results.append({
        "Dataset": name,
        "Label":   label_col,
        "Acc":     metrics["Acc"],
        "Rec_o":   metrics["Rec_o"],
        "Rec_d":   metrics["Rec_d"],
        "Train":   df_train.count(),
        "Test":    df_test.count(),
    })
    print(f"  ✅ Done !")

# Libérer à la fin
if model is not None:
    del model

In [0]:
# ============================================================
# CELLULE 6 — Tableau comparatif avec le papier
# Table résultats Section 5
# ============================================================

print("\n" + "="*70)
print("   RÉSULTATS — Comparaison avec le papier (Section 5)")
print("="*70)
print(f"{'Dataset':<12} {'Acc':>7} {'Rec_o':>7} {'Rec_d':>7} "
      f"| {'Acc(p)':>7} {'Rec_d(p)':>9}")
print("-"*70)

# Valeurs de référence du papier (th=15 et th=60, dataset D2)
papier_ref = {
    "D2_th15": {"Acc": 74.2, "Rec_d": 71.8},
    "D2_th60": {"Acc": 85.8, "Rec_d": 86.9},
}

for r in all_results:
    ref = papier_ref.get(r["Dataset"], {})
    acc_p   = ref.get("Acc",   "—")
    recd_p  = ref.get("Rec_d", "—")
    print(f"{r['Dataset']:<12} "
          f"{r['Acc']:>6}% {r['Rec_o']:>6}% {r['Rec_d']:>6}% "
          f"| {str(acc_p):>6}% {str(recd_p):>8}%")

print("="*70)
print("(p) = valeurs papier Belcastro et al. 2016")

# ============================================================
# CELLULE 7 — Sauvegarder résultats CSV
# ============================================================

df_results = spark.createDataFrame(all_results)
df_results.coalesce(1).write.mode("overwrite").csv(
    f"{OUTPUT}ml_results/",
    header=True
)
print(f"\n✅ Résultats sauvegardés → {OUTPUT}ml_results/")
print("\n✅ Notebook 04 TERMINÉ !")

In [0]:
import matplotlib.pyplot as plt
import numpy as np

# ── Résultats obtenus ──
datasets = ["D1_th15", "D2_th15", "D3_th15",
            "D1_th60", "D2_th60", "D3_th60"]

acc   = [63.1, 64.4, 61.9, None, None, None]
rec_o = [76.9, 73.1, 68.9, None, None, None]
rec_d = [49.3, 55.6, 54.9, None, None, None]

# ── Valeurs papier (D2 uniquement) ──
papier_acc  = {"D2_th15": 74.2, "D2_th60": 85.8}
papier_recd = {"D2_th15": 71.8, "D2_th60": 86.9}

x     = np.arange(len(datasets))
width = 0.25

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "Comparaison des métriques RF — Notre implémentation vs Papier",
    fontsize=13, fontweight="bold"
)

colors = {
    "Acc":    "#1f77b4",
    "Rec_o":  "#2ca02c",
    "Rec_d":  "#ff7f0e",
    "Papier": "#d62728",
}

for ax_idx, (threshold, indices) in enumerate(
    [("th=15", [0,1,2]), ("th=60", [3,4,5])]
):
    ax   = axes[ax_idx]
    lbls = [datasets[i] for i in indices]
    xi   = np.arange(len(lbls))

    # Barres métriques
    vals_acc   = [acc[i]   if acc[i]   is not None else 0 for i in indices]
    vals_rec_o = [rec_o[i] if rec_o[i] is not None else 0 for i in indices]
    vals_rec_d = [rec_d[i] if rec_d[i] is not None else 0 for i in indices]

    b1 = ax.bar(xi - width, vals_acc,   width, label="Acc",   color=colors["Acc"],   alpha=0.85)
    b2 = ax.bar(xi,         vals_rec_o, width, label="Rec_o", color=colors["Rec_o"], alpha=0.85)
    b3 = ax.bar(xi + width, vals_rec_d, width, label="Rec_d", color=colors["Rec_d"], alpha=0.85)

    # Lignes papier D2
    d2_key = f"D2_{threshold}"
    if d2_key in papier_acc:
        d2_idx = lbls.index(d2_key)
        ax.hlines(papier_acc[d2_key],
                  d2_idx - 1.5*width, d2_idx + 1.5*width,
                  colors=colors["Papier"], linewidths=2,
                  linestyles="--", label=f"Acc papier ({d2_key})")
        ax.hlines(papier_recd[d2_key],
                  d2_idx - 1.5*width, d2_idx + 1.5*width,
                  colors="purple", linewidths=2,
                  linestyles=":", label=f"Rec_d papier ({d2_key})")

    # Valeurs sur les barres
    for bar in [b1, b2, b3]:
        for rect in bar:
            h = rect.get_height()
            if h > 0:
                ax.text(
                    rect.get_x() + rect.get_width()/2,
                    h + 0.5, f"{h:.1f}",
                    ha="center", va="bottom", fontsize=7.5
                )

    ax.set_title(f"Threshold {threshold}", fontsize=11)
    ax.set_xticks(xi)
    ax.set_xticklabels(lbls, rotation=15, fontsize=9)
    ax.set_ylim(0, 100)
    ax.set_ylabel("Score (%)")
    ax.legend(fontsize=8)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.spines[["top","right"]].set_visible(False)

    # Annoter les non-évalués (th=60)
    if threshold == "th=60":
        for xi_val, lbl in zip(xi, lbls):
            ax.text(xi_val, 5, "N/A\n(contrainte\nmémoire)",
                    ha="center", va="bottom",
                    fontsize=7, color="gray", style="italic")

plt.tight_layout()
plt.savefig("ml_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Graphique sauvegardé → ml_results.png")

In [0]:
# CELLULE 8 — Analyse de scalabilité

import time
import matplotlib.pyplot as plt
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler, Imputer

OUTPUT    = "/Volumes/workspace/default/outputs/"

METEO_VARS_NUM = ["Temp", "WSpd", "Vis"]
SLOTS          = range(3)
FEATURES_ORIG  = [f"orig_{v}_{i}h" for i in SLOTS for v in METEO_VARS_NUM]
FEATURES_DEST  = [f"dest_{v}_{i}h" for i in SLOTS for v in METEO_VARS_NUM]
ALL_FEATURES   = FEATURES_ORIG + FEATURES_DEST

def build_pipeline(label_col, features):
    imputer = Imputer(
        inputCols=features,
        outputCols=[f"{c}_imp" for c in features],
        strategy="mean"
    )
    assembler = VectorAssembler(
        inputCols=[f"{c}_imp" for c in features],
        outputCol="features",
        handleInvalid="skip"
    )
    rf = RandomForestClassifier(
        labelCol=label_col,
        featuresCol="features",
        numTrees=10,
        maxDepth=5,
        seed=42
    )
    return Pipeline(stages=[imputer, assembler, rf])

fractions = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
times     = []
model     = None

df_train_full = spark.read.parquet(f"{OUTPUT}D4_th15_train/")
df_test_full  = spark.read.parquet(f"{OUTPUT}D4_th15_test/")

for frac in fractions:
    df_sample = df_train_full.sample(
        withReplacement=False, fraction=frac, seed=42
    )
    existing = df_sample.columns
    features = [f for f in ALL_FEATURES if f in existing]

    pipeline = build_pipeline("label_15", features)

    if model is not None:
        del model

    start   = time.time()
    model   = pipeline.fit(df_sample)
    model.transform(df_test_full).count()
    elapsed = time.time() - start

    n = df_sample.count()
    times.append(elapsed)
    print(f"  fraction={frac:.1f} | n={n:,} | temps={elapsed:.1f}s")

if model is not None:
    del model

# ── Graphique ──
plt.figure(figsize=(7, 4))
plt.plot(
    [f * 100 for f in fractions], times,
    marker="o", linewidth=2,
    color="#1f77b4", markersize=7
)
for f, t in zip(fractions, times):
    plt.annotate(
        f"{t:.1f}s",
        xy=(f*100, t),
        xytext=(4, 6),
        textcoords="offset points",
        fontsize=8
    )
plt.xlabel("Fraction des données d'entraînement (%)")
plt.ylabel("Temps d'exécution (secondes)")
plt.title("Scalabilité — Temps d'exécution vs volume\n"
          r"Dataset $D_4$ th=15")
plt.grid(linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("scalability.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Graphique sauvegardé → scalability.png")